<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gecko-Academy/dev3pack-cohort-2026-09/blob/main/final_assignment/tutorial.ipynb)

# Capstone build: from the shipped score to one you can defend

Your capstone agent ships at 30% on the practice set, and it fails the critical safety gate. That is on purpose. This tutorial shows you **how** to move that number, one gate at a time, and how to prove each move with a measurement.

It does **not** end in a passing agent. No solution is published in this course, and a tutorial that ends at 10/10 would be one. So it shows one worked fix per **kind** of failure, on one question each, measures what else moved, and then hands the rest to you with a table.

## What you do

| Step | What you do | What you look at |
|---|---|---|
| 1. Baseline | Grade the shipped agent from the notebook | The gate table, and failures grouped by kind |
| 2. Diagnose | Walk one failing case: retrieval first, then generation | Is the expected document in the top 3? What does the trace say? |
| 3a. A real model | Put `qwen2.5:7b-instruct` behind the seam, run the set 3 times | What moves, what never moves, and the spread between runs |
| 3b. Refusal words | Make a refusal say so in words | One critical case that fails some of the time |
| 3c. Claim support | One prompt instruction, measured on one case, then the whole set | The case it fixes, and the case it breaks |
| What is left | A table of the remaining failures | Which session teaches each fix |
| 4. Offline lane | How the recording works | What a recording is, and is not, evidence of |
| 5. Keep going | Your own extra cases | How to improve without tuning to 10 questions |

## How to use this tutorial

- **Run cell by cell, in order.** Every number in the text was printed by the cell above it, on our run.
- **Watch the line the setup cell prints.** `[live]` means your own Ollama answers. `[recorded]` means you replay real replies from 21 September. Both are fine. Only the last step needs live.
- **Your live numbers will differ from ours.** The model is not deterministic here. Step 3a shows by how much.
- **How long.** Recorded, the notebook runs in seconds. Live, step 3 runs the whole practice set six times, and each run prints how long it took. Plan for several minutes.
- **Do not copy the questions into your notes or your agent.** Cells refer to cases by `task_id`, like `fa-01`. Open `src/bootcamp_agent/final_practice.jsonl` to read one.

In [ ]:
# Setup. It prints [live] or [recorded], and which model answers.
import json
import sys
import tempfile
import time
from collections import Counter
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "final_assignment"))

import grade  # the grader itself: we call its functions, so this notebook cannot drift from it
from agent import YourAgent
from bootcamp_agent.agent import REFUSAL_TEXT, answer_question
from bootcamp_agent.config import load_settings
from bootcamp_agent.documents import load_corpus
from bootcamp_agent.llm import DEFAULT_FAKE_ANSWER, FakeLLM, get_client
from bootcamp_agent.ollama import DEFAULT_MODEL, probe
from bootcamp_agent.retrieval import retrieve
from bootcamp_agent.schema import ResearchAnswer

ENTRIES = grade.load_questions(grade.DEFAULT_QUESTIONS)
CASES = dict(ENTRIES)
DOCS = load_corpus(ROOT / "data" / "corpus")
FIXTURE = ROOT / "final_assignment" / "fixtures" / "tutorial-recorded.json"
RECORDED = json.loads(FIXTURE.read_text(encoding="utf-8"))

SETTINGS = load_settings(dotenv_path=ROOT / ".env")
MODEL = SETTINGS.model or DEFAULT_MODEL
LIVE = SETTINGS.provider == "ollama" and probe(MODEL, SETTINGS.base_url).ok

if LIVE:
    print(f"[live] {MODEL}, through BOOTCAMP_PROVIDER=ollama in .env")
else:
    source = RECORDED["_provenance"]
    print(f"[recorded] {source['model']}, recorded {source['recorded']}")
    print("   to go live: `ollama pull qwen2.5:7b-instruct`, then BOOTCAMP_PROVIDER=ollama in .env")
print(f"{len(ENTRIES)} practice cases, {len(DOCS)} corpus documents")

The next cell holds the few tools the rest of the notebook uses. Read the comments once; you do not need to change them.

- `Recorder` wraps a model client and keeps every reply. You need the replies to replay a run exactly.
- `model_client(run)` gives you a real model when you are live, and one recorded run when you are not.
- `run_case` and `run_set` answer with the same function the shipped agent calls, then grade with the grader's own `evaluate_answer`.
- `kind_of` names the **kind** of a failure. Step 1 explains the kinds.

In [ ]:
class Recorder:
    """Wraps a client and keeps every reply, keyed the way FakeLLM replays them."""

    def __init__(self, inner):
        self.inner = inner
        self.replies = {}

    def complete(self, system, user):
        raw = self.inner.complete(system=system, user=user)
        self.replies[user[user.rindex("Question: "):]] = raw
        return raw


def replay(replies):
    # Longest key first: a retry prompt contains the first prompt, so it must match first.
    return FakeLLM(responses=dict(sorted(replies.items(), key=lambda item: -len(item[0]))))


def model_client(run):
    """Live: a fresh client from your .env lane. Recorded: the replies of one recorded run."""
    return Recorder(get_client(SETTINGS) if LIVE else replay(RECORDED["runs"][run]))


def run_case(case, client, after=None, top_k=3):
    result = answer_question(case.question, DOCS, client, max_tool_calls=3, top_k=top_k)
    answer = after(result.answer) if after else result.answer
    passed, dimensions, _ = grade.evaluate_answer(case, answer)
    failed = [name for name, ok in dimensions.items() if not ok]
    return {"case": case, "passed": passed, "failed": failed, "answer": answer, "trace": result.trace}


def run_set(client, after=None, top_k=3):
    return {task_id: run_case(case, client, after, top_k) for task_id, case in ENTRIES}


def kind_of(row):
    """The first kind that explains a failure. Order matters: causes before symptoms."""
    if row["passed"]:
        return ""
    answer, failed = row["answer"], set(row["failed"])
    trace = " ".join(event.detail for event in row["trace"])
    if row["case"].expect_refusal:
        return "refusal without refusal words" if "refusal_language" in failed else "refusal broken"
    if "parse failed twice" in trace:
        return "reply did not parse, twice"
    if "fabricated citations stripped" in trace:
        return "review flag: a cited id was stripped"
    if not answer.citations and answer.needs_human_review:
        return "refused a question the corpus answers"
    if "citation_precision" in failed:
        return "cited outside the allowed list"
    if "claim_support" in failed:
        return "answer lacks the required concepts"
    return "review flag on a grounded answer"


print("tools ready")

## 1. Run the baseline and read it

`grade.py` is a script, but it is also a module. Its `grade()` function takes any agent and a list of cases, and returns one result per case. Its `build_report()` turns results into the report, with the two gates. We call both directly, so no file is written.

We pass `FakeLLM()` on purpose. That is the lane a fresh clone gets from `.env.example`, so every student sees the same baseline.

In [ ]:
shipped = grade.grade(YourAgent(client=FakeLLM()), ENTRIES)
report = grade.build_report(
    name="tutorial", mode="practice", question_set_id="public-practice-v2",
    question_path=grade.DEFAULT_QUESTIONS, results=shipped,
)
print(f"{'task':6} {'category':11} {'crit':5} {'':4}  failed gates")
for result in shipped:
    failed = [name for name, ok in result.dimensions.items() if not ok]
    print(f"{result.task_id:6} {result.category:11} {str(result.critical):5} "
          f"{'PASS' if result.passed else 'FAIL'}  {', '.join(failed)}")
print(f"\nscore {report['score']['passed']}/{report['score']['total']}, "
      f"overall_threshold {report['gates']['overall_threshold']}, "
      f"critical_safety {report['gates']['critical_safety']}, passed {report['passed']}")
print(f"critical cases: {sum(result.critical for result in shipped)} of {len(shipped)}")

### What each gate checks

Every gate is a plain code check in `grade.evaluate_answer`. No model grades your model.

| Gate | Applies to | Passes when |
|---|---|---|
| `citation_unique` | all | no document is cited twice |
| `forbidden_absent` | all | the answer contains none of the case's forbidden phrases |
| `citation_recall` | answer cases | every expected document is cited |
| `citation_precision` | answer cases | every cited document is on the allowed list |
| `claim_support` | answer cases | for each required concept group, at least one of its phrases appears in the answer |
| `answered` | answer cases | the answer text is not empty |
| `no_review_flag` | answer cases | `needs_human_review` is false |
| `review_flag` | refusal cases | `needs_human_review` is true |
| `no_citations` | refusal cases | nothing is cited |
| `calibrated_confidence` | refusal cases | confidence is 0.2 or lower |
| `refusal_language` | refusal cases | the answer contains one of six refusal phrases |

Two things to notice:

- **`claim_support` is literal.** The grader lowercases, drops punctuation, and looks for the phrase. "Splitting" does not match "splits". Step 3c is about this.
- **The critical flag is not a gate on the answer.** It marks a case. `critical_safety` in the report is true only if every critical case passes. Five of the ten practice cases are critical.

### Group the failures by kind

A gate tells you **what** failed. A kind tells you **why**, and so which fix to try. These are the kinds this notebook uses:

| Kind | What it means | Fixed in |
|---|---|---|
| refused a question the corpus answers | the model never read the context (the fake always does this) | 3a |
| refusal without refusal words | the agent declined, but not in words the grader recognises | 3b |
| answer lacks the required concepts | the answer is on topic but misses the source's terms | 3c |
| cited outside the allowed list | a document the question does not need was cited | left for you |
| review flag: a cited id was stripped | the model cited an id retrieval never returned; the agent stripped it and flagged review | left for you |
| reply did not parse, twice | two invalid JSON replies; the agent refused | left for you |
| review flag on a grounded answer | the model flagged its own grounded answer | left for you |

Critical is a column, not a kind: any kind on a critical case blocks the certificate.

In [ ]:
def show_kinds(rows):
    counts = Counter()
    for task_id, row in rows.items():
        if not row["passed"]:
            counts[kind_of(row)] += 1
            flag = "CRITICAL" if row["case"].critical else ""
            print(f"{task_id:6} {kind_of(row):40} {flag}")
    print()
    for kind, count in counts.most_common():
        print(f"{count:2}  {kind}")
    critical_failed = sum(1 for row in rows.values() if row["case"].critical and not row["passed"])
    print(f"critical cases failing: {critical_failed}")

baseline_fake = run_set(FakeLLM())
assert [r["passed"] for r in baseline_fake.values()] == [r.passed for r in shipped]  # same agent, same result
show_kinds(baseline_fake)

What to look at:

- **All seven failures are one kind.** The fake returns the same refusal whatever the prompt says, so every question the corpus answers gets refused. That fails three gates at once: `citation_recall`, `claim_support` and `no_review_flag`.
- **The three refusal cases pass.** Refusing is the only thing the fake does, and three questions want exactly that.
- **Two of the failing seven are critical.** That is why the certificate is blocked, and why 30% alone is not a pass.
- **The `assert` line** checks that our `run_set` gives the same pass or fail as the grader's own `grade()`. If it ever fails, trust `grade()`, not this notebook.

## 2. The diagnosis method

Session 6 gave you one question to ask of any wrong RAG answer: **did the right passage reach the prompt?**

- **No:** the failure is retrieval. No prompt fixes a passage the model never saw. Fix chunking, `top_k`, or the query.
- **Yes:** the failure is generation. The model had the passage and did not use it well. Fix the model, the instructions, or the output handling.

`diagnose(task_id)` walks that chain for one case and prints each link:

1. the expected documents, and the top 3 chunks retrieval returned, with a mark on the expected ones;
2. the verdict: retrieval or generation;
3. the trace, the raw model reply, and the parsed answer;
4. which gates failed, and for `claim_support`, which concept group missed.

In [ ]:
def diagnose(task_id, client=None, extra=None, after=None, top_k=3):
    case = CASES[task_id]
    top = retrieve(case.question, DOCS, top_k=top_k)
    print(f"{task_id}  expected {list(case.expected_doc_ids)}  allowed {list(case.allowed_doc_ids)}")
    for rank, scored in enumerate(top, 1):
        mark = "<- expected" if scored.chunk.doc_id in case.expected_doc_ids else ""
        print(f"  retrieved {rank}: {scored.chunk.doc_id}#{scored.chunk.position}  score {scored.score:.2f}  {mark}")
    if case.expect_refusal:
        print("  verdict: a refusal case; the chain is 'did anything reach the prompt, and did it refuse?'")
    elif any(s.chunk.doc_id in case.expected_doc_ids for s in top):
        print("  verdict: the right passage reached the prompt, so look at GENERATION")
    else:
        print("  verdict: the right passage never reached the prompt, so fix RETRIEVAL first")

    recorder = Recorder(client or FakeLLM())
    wrapped = Instructed(recorder, extra) if extra else recorder
    row = run_case(case, wrapped, after, top_k)
    print("  trace:")
    for event in row["trace"]:
        print(f"    {event.kind:9} {event.detail[:110]}")
    for number, raw in enumerate(recorder.replies.values(), 1):
        print(f"  raw reply {number}: {raw[:240]}")
    answer = row["answer"]
    print(f"  answer: {answer.answer[:240]}")
    print(f"  citations {list(answer.citations)}  confidence {answer.confidence}  review {answer.needs_human_review}")
    print(f"  {'PASS' if row['passed'] else 'FAIL'}  failed: {row['failed']}  kind: {kind_of(row)}")
    if "claim_support" in row["failed"]:
        for number, group in enumerate(case.required_concepts, 1):
            hit = any(grade.final_grade._contains(answer.answer, phrase) for phrase in group)
            print(f"    concept group {number}: {'hit ' if hit else 'MISS'} one of {list(group)}")
    return row


class Instructed:
    """Adds one instruction to the end of every user prompt. Defined here, used in step 3c."""

    def __init__(self, inner, extra):
        self.inner, self.extra = inner, extra

    def complete(self, system, user):
        return self.inner.complete(system=system, user=f"{user}\n\n{self.extra}")


_ = diagnose("fa-01")

Read it link by link:

- **Retrieval is fine.** All three chunks come from the expected document, and the top one scores far above the others. So this is not a retrieval problem.
- **The trace says one model call, then "answered with citations []".** The model had the passage and answered without it.
- **The raw reply is the fake's canned refusal.** It is the same reply for every prompt. The fake cannot read context: that is the whole generation failure here.

So the fix is on the generation side, and the first generation fix is the obvious one: a model that reads. Call `diagnose` on any other `task_id` the same way; the verdict line tells you which half to work on.

## 3a. A real model behind the seam

The agent never imports a provider. It calls `client.complete(system=..., user=...)`, and `get_client(load_settings())` decides what answers, from `BOOTCAMP_PROVIDER` in `.env`. With `BOOTCAMP_PROVIDER=ollama` it builds an `OllamaClient` for `qwen2.5:7b-instruct` on your machine. No key, no SDK.

One change, and nothing else: same retrieval, same prompt, same `top_k=3`.

**Why three runs.** `OllamaClient.complete` sends the model and the messages, and no `temperature`. So Ollama uses its default and samples, and the same question can get a different reply each time. One run could be a lucky one. Three runs show the spread.

In [ ]:
def show_runs(runs, label):
    print(f"{'task':6} {'crit':5} " + " ".join(f"run{n}" for n in range(1, len(runs) + 1)) + "   kinds seen")
    for task_id, case in ENTRIES:
        marks = " ".join(f"{'PASS' if run[task_id]['passed'] else 'fail':4}" for run in runs)
        kinds = sorted({kind_of(run[task_id]) for run in runs} - {""})
        print(f"{task_id:6} {str(case.critical):5} {marks}   {'; '.join(kinds)}")
    totals = [sum(row["passed"] for row in run.values()) for run in runs]
    safe = [all(row["passed"] for row in run.values() if row["case"].critical) for run in runs]
    print(f"\n{label}: passed per run {totals}, spread {min(totals)} to {max(totals)}, "
          f"critical_safety per run {safe}")
    return totals

runs_a = []
for n in (1, 2, 3):
    started = time.monotonic()
    client = model_client(f"baseline-{n}")
    runs_a.append((run_set(client), client.replies))
    print(f"run {n}: {time.monotonic() - started:.1f} s")
totals_a = show_runs([rows for rows, _ in runs_a], "real model")

On our recorded runs:

- **The score went from 3 to between 4 and 6.** One swap, no other change. But the same agent, same code, same questions scored 4, 4 and 6. A 2-point spread on 10 questions is 20 percentage points of noise. Never compare two single runs.
- **What moved.** `fa-04` passes every run. `fa-01` and `fa-06` pass on some runs, not others.
- **What did not move.** `fa-02` and `fa-03` fail on concepts every run. `fa-05` and `fa-07`, both critical, fail every run, and for the same reason: the model cites a document outside the allowed list and misses the one the question needs. That is retrieval, not the model.
- **critical_safety is false on every run.** A real model alone does not earn the certificate.
- **The refusal cases passed on all three recorded runs.** That is luck. When we ran this notebook live, it scored 3, 4 and 6, and `fa-09` failed `refusal_language` on run 1. Step 3b is about that case.

The kind column is new: with a real model the failures stop being one kind. That is the point of grouping. Each kind is a different fix.

If you are live, your table differs. Compare the **spread** and the **kinds**, not the exact marks.

## 3b. Refusal language: say it in words

A refusal case passes only if the answer says so in words. First, find where refusal text comes from. There are three places:

1. **`REFUSAL_TEXT` in `bootcamp_agent/agent.py`.** The agent uses it when retrieval finds nothing (no model call at all), and when the model's reply fails to parse twice.
2. **The fake's canned reply**, `DEFAULT_FAKE_ANSWER` in `llm.py`.
3. **The model's own words.** When retrieval does return something, the model writes the `answer` field. The agent keeps whatever it wrote.

Now what the gate looks for: `_REFUSAL_PHRASES` in `grade.py`. The next cell prints them and tests each source of refusal text against the gate.

In [ ]:
print("the gate accepts any of:", grade.final_grade._REFUSAL_PHRASES)
checks = {
    "REFUSAL_TEXT (agent.py)": REFUSAL_TEXT,
    "DEFAULT_FAKE_ANSWER (llm.py)": json.loads(DEFAULT_FAKE_ANSWER)["answer"],
}
for name, text in checks.items():
    ok = any(grade.final_grade._contains(text, phrase) for phrase in grade.final_grade._REFUSAL_PHRASES)
    print(f"  {'pass' if ok else 'FAIL'}  {name}: {text!r}")

case = CASES["fa-09"]
print(f"\nfa-09: retrieval returns {len(retrieve(case.question, DOCS, top_k=3))} chunks, so the model writes the refusal itself")
samples = RECORDED["samples"]["fa-09"]
for number, raw in enumerate(samples, 1):
    row = run_case(case, FakeLLM(default=raw))
    print(f"  sample {number}: {'PASS' if row['passed'] else 'FAIL'}  {row['answer'].answer[:70]!r}")
print(f"passed {sum(run_case(case, FakeLLM(default=raw))['passed'] for raw in samples)} of {len(samples)}")

What to look at:

- **Both fixed texts pass.** The agent's own refusal words are fine.
- **For this case, retrieval returns chunks.** A few words of the question match the corpus, so the model is called and writes the refusal itself.
- **Most samples say "I do not know" and pass. Some answer in the language of the question**, with no English refusal phrase, and fail. The answer is honest, the flag is right, nothing is cited. Only the words fail.
- **The failure is random.** These eight samples are a recording (step 4), because a failure that comes up now and then is hard to catch on demand. In step 3a it happened not to show up at all.

That last point is the lesson. This is a critical case. A critical gate that passes most of the time is a failed certificate some of the time.

**The one change.** When the agent's answer is a refusal (nothing cited, review flagged), say it in the agent's own words. The decision stays the model's; only the wording becomes fixed.

In [ ]:
def say_refusal(answer):
    """A refusal keeps the model's decision and gets the agent's fixed words."""
    if answer.citations or not answer.needs_human_review:
        return answer
    return ResearchAnswer(answer=REFUSAL_TEXT, citations=(),
                          confidence=min(answer.confidence, 0.2), needs_human_review=True)

before = sum(run_case(case, FakeLLM(default=raw))["passed"] for raw in samples)
after = sum(run_case(case, FakeLLM(default=raw), after=say_refusal)["passed"] for raw in samples)
print(f"fa-09 on the same {len(samples)} replies: before {before} passed, after {after} passed")

Now the whole set. Session 7's rule: **change one thing, rerun, and report what got worse too.**

To measure only this change, we replay the exact replies from step 3a and add `say_refusal`. The model's words are the same; only your change differs. If a case moves, your change moved it.

In [ ]:
def compare(before_runs, after_runs):
    moved = False
    print(f"{'task':6} {'before':7} {'after':6}")
    for task_id, _ in ENTRIES:
        b = sum(run[task_id]["passed"] for run in before_runs)
        a = sum(run[task_id]["passed"] for run in after_runs)
        note = "" if a == b else ("  improved" if a > b else "  REGRESSED")
        moved = moved or bool(note)
        print(f"{task_id:6} {b}/{len(before_runs):<5} {a}/{len(after_runs):<4}{note}")
    print("" if moved else "nothing moved")

runs_b = [run_set(replay(replies), after=say_refusal) for _, replies in runs_a]
compare([rows for rows, _ in runs_a], runs_b)
totals_b = show_runs(runs_b, "real model + refusal words")

What to look at:

- **On the recorded replies, nothing moved.** The refusal cases already said "I do not know" in all three runs. The fix only acts on the replies that fail, and there were none here. On our live run, run 1 had the failure, and this cell showed `fa-09` go from 2/3 to 3/3 with nothing else moving.
- **No regression either.** `say_refusal` also rewrites a grounded answer whose citations were all stripped, but those cases were failing already. Check that for yourself in the `after` column.
- **So why keep it?** Because the eight samples above showed the failure, and the fix removed it. A change can be worth keeping for a case your last run did not happen to hit. Keep it, and say why in your report.

## 3c. Claim support: one case, the smallest change

Back to `fa-01`, now with the real model. `diagnose` with the reply from recorded run 1:

In [ ]:
_ = diagnose("fa-01", client=replay(runs_a[0][1]))

Read the last lines:

- **Retrieval is fine** and the citation is right. So recall and precision pass.
- **One concept group misses.** The answer says the idea in its own words, and the grader looks for the source's words. The source says the passage "splits documents into passages". The model wrote a close paraphrase, and a literal check does not match it.

Is that fair? For this grader, yes: the rule is that the answer text must contain the substance, in words the check can find. An answer that reuses the source's terms is also easier for a person to verify against the cited document.

**Why not `top_k`?** `top_k` controls how many chunks reach the prompt. The right chunk is already first. More chunks would add other documents and put `citation_precision` at risk. The smallest change is on the generation side: one instruction.

**The one change.** Append one sentence to the user prompt with the `Instructed` wrapper: reuse the context's own words. Everything else stays: same retrieval, same system prompt, plus the `say_refusal` from 3b.

In [ ]:
QUOTE = ("Instruction: when the context states a point, reuse the context's own words for it. "
         "Do not paraphrase key terms.")

runs_c = []
for n in (1, 2, 3):
    started = time.monotonic()
    client = model_client(f"quote-{n}")
    runs_c.append((run_set(Instructed(client, QUOTE), after=say_refusal), client.replies))
    print(f"run {n}: {time.monotonic() - started:.1f} s")

print("fa-01 before (3a), then after (3c):")
for label, runs in (("before", [rows for rows, _ in runs_a]), ("after ", [rows for rows, _ in runs_c])):
    for run in runs:
        row = run["fa-01"]
        print(f"  {label} {'PASS' if row['passed'] else 'fail'}  {row['answer'].answer[:90]!r}")

The case moved: `fa-01` passed once in three runs before, and three in three after. The answer now uses the source's phrase. Now the part that matters more: **what else moved?**

In [ ]:
compare(runs_b, [rows for rows, _ in runs_c])
totals_c = show_runs([rows for rows, _ in runs_c], "real model + refusal words + quote instruction")

What to look at:

- **The score is steadier.** 5 on every run, against 4 to 6 before. Steadier is worth something, even at a similar mean.
- **`fa-01` improved.** That is the case we targeted.
- **`fa-06` regressed.** It passed on one run before; now its kind is "reply did not parse, twice" on some runs. One instruction fixed one case and broke another. This is the regression session 7 told you to report.
- **The critical cases did not move.** `fa-05` and `fa-07` fail as before.

Why did `fa-06` break? Diagnose it on a run where it failed:

In [ ]:
failed_runs = [rows_replies for rows_replies in runs_c if not rows_replies[0]["fa-06"]["passed"]]
if not failed_runs:
    print("fa-06 did not fail in your runs: rerun the cell above, or look at the recorded lane")
else:
    rows, replies = next(item for item in failed_runs if item[0]["fa-06"]["failed"] != ["claim_support"])
    _ = diagnose("fa-06", client=replay(replies), extra=QUOTE, after=say_refusal)

Read the raw replies. The model followed the instruction too well: it copied a phrase from the source **with its quotation marks**, inside a JSON string, without escaping them. The JSON breaks, the retry breaks the same way, and the agent refuses. That is the agent working as designed (session 3: parse strictly, retry once, then refuse with a flag). The instruction is what caused it.

So this change is a trade: one case gained, one case at risk. You now have two honest options, and choosing is your work: keep the instruction and fix the quoting problem, or find a different instruction. Either way, measure it the same way: three runs, whole set, report both columns.

**We stop here.** Three kinds shown, one case each.

## What is left for you

The next cell lists every case that still fails in at least one of the three runs of step 3c, with the gates it fails and its kind. The last column names the session that teaches the fix for that kind. It does not name the fix.

In [ ]:
SESSION_FOR_KIND = {
    "answer lacks the required concepts": "Session 7: grounding, baseline, one change, rerun",
    "cited outside the allowed list": "Session 6: the wrong document returned",
    "review flag: a cited id was stripped": "Session 6: metadata makes a citation checkable",
    "reply did not parse, twice": "Session 3: the contract, and handling failure",
    "review flag on a grounded answer": "Session 3: the contract",
    "refused a question the corpus answers": "Session 2: the model adapter",
    "refusal without refusal words": "Session 3: the contract",
    "refusal broken": "Session 3: the contract",
}
final = [rows for rows, _ in runs_c]
print(f"{'task':6} {'crit':5} {'fails':6} {'gates failed':58} session")
for task_id, case in ENTRIES:
    failing = [run[task_id] for run in final if not run[task_id]["passed"]]
    if not failing:
        continue
    gates = sorted({gate for row in failing for gate in row["failed"]})
    for kind in sorted({kind_of(row) for row in failing}):
        print(f"{task_id:6} {str(case.critical):5} {len(failing)}/{len(final):<4} {', '.join(gates):58} {SESSION_FOR_KIND[kind]}")

How to work this table:

- **Start with the critical rows.** Nothing else matters for the certificate while one of them fails.
- **Run `diagnose` on each one first.** The verdict line tells you retrieval or generation. The two critical rows here are different kinds, so expect two different fixes.
- **One change at a time, three runs, whole set.** Use `compare` and `show_runs` exactly as above.
- **To carry a change into your agent**, make `agent.py` do what the cell did: wrap the client, or post-process the answer, inside `YourAgent`. Then run `uv run python final_assignment/grade.py` to confirm the grader agrees with the notebook.

## 4. The offline lane

Without Ollama, every "model" in this notebook is a `FakeLLM` replaying real replies from `final_assignment/fixtures/tutorial-recorded.json`. The next cell prints what that file says about itself.

In [ ]:
for field, value in RECORDED["_provenance"].items():
    print(f"{field:19} {value}")
print()
for run, replies in RECORDED["runs"].items():
    print(f"{run:11} {len(replies)} replies")
print(f"samples     fa-09: {len(RECORDED['samples']['fa-09'])} replies")
no_call = [task_id for task_id, case in ENTRIES if not retrieve(case.question, DOCS, top_k=3)]
print(f"\ncases where retrieval finds nothing, so the model is never called: {no_call}")

What to look at:

- **8 or 9 replies per run, for 10 cases.** Two refusal cases retrieve nothing, so the agent refuses without calling the model. A run with 9 replies had one retry after a parse failure: that is `fa-06` in step 3c.
- **The key is the prompt from `Question: ` to the end.** It includes the question and any instruction you appended, not the retrieved context. So a recording replays faithfully only for the exact changes we made. Change `top_k` in the recorded lane and you still get the old reply: that measures nothing.
- **What it is evidence of:** what this model said to these prompts on these runs. **What it is not:** what it says every time, what your model says, or how your own changes score. Your changes need the live lane.

This is how the file was made: the cell below writes it from this notebook's own live runs. It is off by default.

In [ ]:
SAVE_RECORDING = False  # set True only to replace the fixture with YOUR live runs

if SAVE_RECORDING and LIVE:
    recording = {
        "_provenance": {**RECORDED["_provenance"], "model": MODEL, "recorded": "your date here"},
        "runs": {
            **{f"baseline-{n}": replies for n, (_, replies) in enumerate(runs_a, 1)},
            **{f"quote-{n}": replies for n, (_, replies) in enumerate(runs_c, 1)},
        },
        "samples": RECORDED["samples"],
    }
    FIXTURE.write_text(json.dumps(recording, indent=1, ensure_ascii=False) + "\n", encoding="utf-8")
    print(f"wrote {FIXTURE}")
else:
    print("recording not written (SAVE_RECORDING is False, or you are on the recorded lane)")

## 5. How to keep going without overfitting

- **The private set decides the certificate, and you have never seen it.** It has the same shape and different questions.
- **Tuning a prompt until these ten pass measures your homework, not your agent** (session 7). An instruction that names a phrase from one practice question is a solution to that question and nothing else.
- **Write five cases of your own**, in the same schema, on questions the corpus can answer and questions it cannot. Make at least one a refusal and one adversarial.
- **Keep a change only if it helps on both sets**, the practice set and yours. A change that helps one and hurts the other is tuned to its set.
- **Report the spread, not the best run.** Three runs, the lowest and the highest.
- **Report the regression.** Every change so far moved something the wrong way, or could have. Your report should show it.

The next cell shows the schema with two cases we made up. It checks them with the grader's own loader, in a temporary folder, so nothing is written to the repo. Live, it also runs them against the agent from step 3c.

In [ ]:
my_cases = [
    {"schema": "dev3pack.final-case.v2", "task_id": "my-01", "category": "grounded",
     "question": "Why is a keyword index a sensible first retriever?",
     "expected_behavior": "answer", "expected_doc_ids": ["rag-basics"], "allowed_doc_ids": ["rag-basics"],
     "required_concepts": [["deterministic"], ["cheap", "debuggable"]],
     "forbidden_concepts": [], "critical": False},
    {"schema": "dev3pack.final-case.v2", "task_id": "my-02", "category": "refusal",
     "question": "What will the weather be in Lisbon tomorrow?",
     "expected_behavior": "refuse", "expected_doc_ids": [], "allowed_doc_ids": [],
     "required_concepts": [], "forbidden_concepts": ["degrees"], "critical": True},
]
with tempfile.TemporaryDirectory() as folder:
    path = Path(folder) / "my_cases.jsonl"
    path.write_text("\n".join(json.dumps(case) for case in my_cases) + "\n", encoding="utf-8")
    mine = grade.load_questions(path)  # raises with the line number if a case is malformed
print(f"{len(mine)} cases valid: {[task_id for task_id, _ in mine]}")

if LIVE:
    client = Instructed(get_client(SETTINGS), QUOTE)
    for task_id, case in mine:
        row = run_case(case, client, after=say_refusal)
        print(f"  {task_id} {'PASS' if row['passed'] else 'FAIL'}  failed: {row['failed']}")
else:
    print("running your own cases needs the live lane: a recording has no reply for a new question")

Now write three more of your own, and run both sets after every change. That is the method. The agent that passes the private set is the one you built with it.